In [1]:
import os
import sys
from os.path import join
import copy

# Add parent directory (arch/) to path so we can import sibling modules
# Handle both cases: notebook run from diagnostics/ or from arch/
current_dir = os.getcwd()
if 'diagnostics' in current_dir:
    arch_dir = os.path.dirname(current_dir)
else:
    arch_dir = current_dir

if arch_dir not in sys.path:
    sys.path.insert(0, arch_dir)
    
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import optim, nn
import torch.multiprocessing as mp
from torch.distributed import init_process_group, destroy_process_group
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from astropy.io import fits
import pyxis.torch as pxt
import normflows as nf

from networks import *
from train import *
import config
from model_registry import load_model_config

model_name = 'ViT-CNN-flow_tf_train'
data_dir = '/ocean/projects/phy250048p/shared/datasets/valid_1m'
fig_dir = '/ocean/projects/phy250048p/shared/figures/'
model_dir = '/ocean/projects/phy250048p/shared/models/'

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
model_cfg = load_model_config(model_name, allow_fallback_current=True)
config.set_model_config(model_cfg)

In [4]:
model = load_model(
    mode=1,
    Model=ForkCNN,
    path=join(model_dir, model_name, f'{model_name}199'),
    strict=True,
    assign=True,
    device=device,
    model_name=model_name,
    networks_root='/ocean/projects/phy250048p/shared/networks',
    use_compile=False,
)

In [5]:
# Get data loader
bs = 100
test_args = list(config.test.values())
test_ds = pxt.TorchDataset(data_dir)
test_dl = DataLoader(test_ds,
                     batch_size=bs,
                     pin_memory=False,
                     shuffle=False,)

In [6]:
snrs = torch.rand((len(test_ds),), device=device)*995 + 5
features = []
model.eval()
with torch.no_grad():
    for i, batch in enumerate(test_dl):
        start = i*bs
        end = min((i+1)*bs, len(test_ds))
        snr = snrs[start:end]
        img = apply_noise(batch['img'].float().to(device), snr, device=device)
        spec = apply_noise(batch['spec'].float().to(device), snr, device=device)
        fp = batch['fib_pos'].float().to(device)
        features.append(model.extract_features(img, spec, fp).cpu())

In [7]:
features = torch.cat(features, dim=0)

In [8]:
z1 = features[:, :512]
z2 = features[:, 512:]

In [9]:
vicreg_loss = VICRegLoss()

In [10]:
total_loss, sim_component, var_component, cov_component, eff_dim_z1, eff_dim_z2 = vicreg_loss(z1, z2, return_components=True)

In [11]:
print(total_loss, sim_component, var_component, cov_component, eff_dim_z1, eff_dim_z2)

tensor(5654.8799) tensor(59.1464) tensor(18.9802) tensor(5576.7534) tensor(2.7833) tensor(1.0545)
